In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

X = pd.read_parquet("../data/processed/luad_X.parquet")
labels = pd.read_csv("../data/processed/luad_labels.csv")

y = labels["label"]

(X.index == labels["sample_id"]).all(), X.shape, labels["sample_type"].value_counts()

(np.True_,
 (574, 20530),
 sample_type
 tumor     515
 normal     59
 Name: count, dtype: int64)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train.shape, X_test.shape, y_train.value_counts(), y_test.value_counts()

((459, 20530),
 (115, 20530),
 label
 1.0    412
 0.0     47
 Name: count, dtype: int64,
 label
 1.0    103
 0.0     12
 Name: count, dtype: int64)

In [9]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
selector_model = LogisticRegression(
    penalty="l1",
    solver="saga",
    class_weight="balanced",
    max_iter=10000,
    C=0.1,
    random_state=42,
)

selector_model.fit(X_train_scaled, y_train)

/Users/adityaanand/dev/cancer-lab/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/adityaanand/dev/cancer-lab/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [ ]:
coefs = selector_model.coef_[0]

ranking = pd.DataFrame({
    "gene": X.columns,
    "coef": coefs,
    "abs_coef": np.abs(coefs),
})

ranking = ranking.sort_values("abs_coef", ascending=False)

ranking.head(30)

,gene,coef,abs_coef
2939,PYCR1,0.355015,0.355015
20363,ETV4,0.244203,0.244203
10326,CD5L,-0.224410,0.224410
9724,ADM2,0.202896,0.202896
2677,LGR4,0.177042,0.177042
7081,ATP10B,0.152263,0.152263
7970,OR6K3,-0.150351,0.150351
14020,EMP2,-0.147496,0.147496
11727,SPP1,0.108235,0.108235
16893,C10orf67,-0.107084,0.107084


In [ ]:
panel_sizes = [20, 10, 5, 3, 2]

def evaluate_panel(selected_genes):
    X_train_panel = X_train[selected_genes]
    X_test_panel = X_test[selected_genes]

    scaler_panel = StandardScaler()
    X_train_panel_scaled = scaler_panel.fit_transform(X_train_panel)
    X_test_panel_scaled = scaler_panel.transform(X_test_panel)

    model = LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=42,
    )

    model.fit(X_train_panel_scaled, y_train)

    y_pred = model.predict(X_test_panel_scaled)
    y_prob = model.predict_proba(X_test_panel_scaled)[:, 1]

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    }

In [ ]:
panel_results = []
panel_genes = []

for k in panel_sizes:
    selected_genes = ranking.head(k)["gene"].tolist()
    metrics = evaluate_panel(selected_genes)

    panel_results.append({
        "panel_size": k,
        **metrics,
    })

    for rank, gene in enumerate(selected_genes, start=1):
        panel_genes.append({
            "panel_size": k,
            "rank": rank,
            "gene": gene,
            "coef": ranking.loc[ranking["gene"] == gene, "coef"].iloc[0],
            "abs_coef": ranking.loc[ranking["gene"] == gene, "abs_coef"].iloc[0],
        })

panel_results_df = pd.DataFrame(panel_results)
panel_genes_df = pd.DataFrame(panel_genes)

panel_results_df

In [ ]:
panel_genes_df

In [ ]:
panel_results_df.to_csv(
    "../data/processed/unconstrained_panel_results.csv",
    index=False,
)

panel_genes_df.to_csv(
    "../data/processed/unconstrained_panel_genes.csv",
    index=False,
)